In [ ]:
from hydra import initialize, compose
from clarification_trees_v3.config.schema import parse_config
from clarification_trees_v3.dataset.dataset import SFTClarificationTreeDataset
import json

# Initialize Hydra config and parse it
with initialize(version_base=None, config_path="../config"):
    raw_cfg = compose(config_name="generate_trees")
    cfg = parse_config(raw_cfg)
    
print(f"Config loaded successfully!\n{cfg.model_dump_json(indent=2)}")


In [ ]:
from clarification_trees_v3 import GENERATED_TREES_PATH
trees_path = GENERATED_TREES_PATH / cfg.paths.data.trees_subpath
assert trees_path.exists(), "Cannot find trees"

# Initialize the SFT Clarification Tree Dataset
# You can filter to get either the top_n nodes per tree or those with an advantage >= threshold
sft_ds = SFTClarificationTreeDataset(
    trees_path=trees_path,
    # top_n=1000,
    advantage_threshold=0.,
    load_images=False # Set to True if you need image objects loaded
)

print(f"Loaded {len(sft_ds)} samples.")


In [ ]:
print(len(sft_ds))

In [ ]:
if len(sft_ds) > 0:
    # Grab a sample
    sample = sft_ds[0]

    # Format the dialog leading up to the target using your specific model configuration
    messages = sample.trajectory.to_messages("qwen-3-vl", use_img_path=True)
    target = sample.target

    print("Target Generation:")
    print("-----------------")
    print(target)
    print(f"(Advantage: {sample.advantage:.4f})")
    print("\nPrompt Dialog:")
    print("--------------")
    print(json.dumps(messages, indent=2))
else:
    print("Dataset is empty!")
